In [2]:
#Import Libraries
import numpy as np
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.svm import SVR
import time
import pickle
import os
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:

base_path = '/content/drive/Project ML'

people = ['Person A', 'Person B', 'Person C', 'Person D', 'Person E', 'Person F']
gestures = ['click', 'pinch', 'swipe', 'wave']

data = {}

for person in people:#loop that adds all the cvs files to the data array
    for gesture in gestures:
        gesture_path = os.path.join(base_path,  person, person, gesture)
        if os.path.exists(gesture_path):
            files = [f for f in os.listdir(gesture_path) if f.endswith('.csv')]
            dfs = [pd.read_csv(os.path.join(gesture_path, f)) for f in files]
            combined_df = pd.concat(dfs, ignore_index=True)
            data[f'{person}_{gesture}'] = combined_df
data
print(os.path.exists('/content/drive/MyDrive'))

True


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

#feature extraction based on the first paper

def preprocess_spectrogram(df):#if the entire column is NaNs, we drop it, otherwise we replace the NaNs with the mean of the column
    matrix = df.dropna(axis=1, how='all').to_numpy()
    matrix = np.where( np.isnan(matrix), np.nanmean(matrix, axis=0), matrix)
    return matrix


def extract_envelopes(matrix):#based on paper we classify the hand gestures based on the envelopes of their micro doppler signature
    middle = matrix.shape[1] // 2
    pos_half = matrix[:, middle:]# Positive Doppler
    neg_half = matrix[:, :middle] # Negative Doppler
    # max value across frequency bins for each time step
    pos_envelope = np.max(pos_half, axis=1)
    neg_envelope = np.max(neg_half, axis=1)
    return pos_envelope, neg_envelope


def build_feature_vector(pos_env, neg_env):
    return np.concatenate([pos_env, neg_env])

X = []
y = []

meta=[]

for gesture in gestures:
    for person in people:
        key = f'{person}_{gesture}'
        df = data.get(key)
        if df is not None and not df.empty:
            matrix = preprocess_spectrogram(df)
            pos_env, neg_env = extract_envelopes(matrix)
            features = build_feature_vector(pos_env, neg_env)
            X.append(features)
            y.append(gesture)
            meta.append((person, gesture))
X
#padding
#def manual_padding(matrix):
   # padded_matrix = np.zeros((matrix.shape[0] + 2, matrix.shape[1] + 2), dtype=matrix.dtype)
   # padded_matrix[1:-1, 1:-1] = matrix
   # return padded_matrix

#max_len = max(len(f) for f in X)
#X_padded = np.array([np.pad(f, (0, max_len - len(f)), 'constant') for f in X])
#X_padded=manual_padding(X)
#scaler = StandardScaler()
#X_scaled = scaler.fit_transform(X_padded)

#X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, stratify=y)

#knn = KNeighborsClassifier(n_neighbors=3, metric='manhattan')

#knn.fit(X_train, y_train)
#print("Accuracy:", knn.score(X_test, y_test))

[]

In [ ]:

X_dict = {person: {gesture: [] for gesture in gestures} for person in people}

for features, (person, gesture) in zip(X, meta):
    X_dict[person][gesture].append(features)

X_dict['Person A']['click']  # list of feature vectors

[]

In [ ]:
#create databases of each gesture and scale them
def scale_data(data):
    scaler = StandardScaler()
    return scaler.fit_transform(data)
#def concatenate_data(gesture):
    #return pd.concat([X_dict['Person A'][gesture], X_dict['Person B'][gesture],X_dict['Person C'][gesture], X_dict['Person D'][gesture], X_dict['Person E'][gesture], X_dict['Person F'][gesture]], ignore_index=True)
#Je pense que c'est pas necessaire en fin de compte d'entrainer
#les gestures séparement, notre but c'est de distinguer les gestures
#dans un modèle, pas de reconnaitre si la data fait partie de cette gesture ou non
click_data= [x for x, label in zip(X, y) if label == 'click']
click_data_scaled =  scale_data(click_data)

pinch_data=[x for x, label in zip(X, y) if label == 'pinch']
pinch_data_scaled = scale_data(pinch_data)

swipe_data=[x for x, label in zip(X, y) if label == 'swipe']
swipe_data_scaled = scale_data(swipe_data)

wave_data=[x for x, label in zip(X, y) if label == 'wave']
wave_data_scaled = scale_data(wave_data)
click_data_scaled

ValueError: Expected 2D array, got 1D array instead:
array=[].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [ ]:
#Task 2
scaler= StandardScaler()

#padding
max_len = max(len(f) for f in X)
X_padded = np.array([np.pad(f, (0, max_len - len(f)), 'constant') for f in X])

X_scaled=scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, stratify=y)
X_knn = KNeighborsClassifier(n_neighbors=3, metric='manhattan')#Knn model according to article

X_knn.fit(X_train, y_train)
print("Accuracy:", knn.score(X_test, y_test))
#SVM
X_SVM=SVR(kernel='linear')
X_SVM.fit(X_train, y_train)
#logistic Regression
X_lin_reg=LinearRegression()
X_lin_reg.fit(X_train, y_train)
#Deep learning models :